# Reliability and descriptive statistics


In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
INPUT_FILE = Path(os.environ.get('EMOTION_RATINGS_FILE', 'data/Sample2372_ratings_English.xlsx'))
RESULTS_ROOT = Path(os.environ.get('EMOTION_RESULTS_ROOT', 'results'))
OUTPUT_DIR = RESULTS_ROOT / '1_reliability_description'
PROCESS_DIR = OUTPUT_DIR / 'process_files'
RESULTS_DIR = OUTPUT_DIR / 'results_files'
for folder in [OUTPUT_DIR, PROCESS_DIR, RESULTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
SEED = 42
N_SPLITS = 10000
TOP_N = 3
EXPECTED_WORDS = 70
EXPECTED_FEATURES = 43
RATING_MIN = 1
RATING_MAX = 9


## Functions


In [ ]:
def save_csv(table, folder, filename, index=False):
    path = folder / filename
    table.to_csv(path, index=index, encoding='utf-8-sig')
    return path

def build_reliability_blocks(data, words):
    word_position = {word: i for i, word in enumerate(words)}
    blocks = {}
    for feature, feature_data in data.groupby('emotion_feature'):
        feature_blocks = []
        for rating_block, block_data in feature_data.groupby('rating_block'):
            matrix = block_data.pivot_table(
                index='participant_id',
                columns='emotion_word',
                values='rating',
                aggfunc='mean'
            )
            if matrix.isna().any().any():
                raise ValueError(
                    f'The rater-by-word matrix for {feature} / {rating_block} contains missing values.'
                )
            word_indices = np.array([word_position[word] for word in matrix.columns])
            feature_blocks.append(
                {
                    'rating_block': rating_block,
                    'word_indices': word_indices,
                    'ratings': matrix.to_numpy(dtype=float)
                }
            )
        blocks[feature] = feature_blocks
    return blocks

def repeated_split_half(blocks, words, features, seed=42, n_splits=10000):
    rng = np.random.default_rng(seed)
    n_words = len(words)
    iteration_rows = []
    summary_rows = []
    for feature_index, feature in enumerate(features, start=1):
        feature_blocks = blocks[feature]
        r_half_values = np.empty(n_splits, dtype=float)
        r_sb_values = np.empty(n_splits, dtype=float)
        for iteration in range(1, n_splits + 1):
            sum_a = np.zeros(n_words, dtype=float)
            count_a = np.zeros(n_words, dtype=float)
            sum_b = np.zeros(n_words, dtype=float)
            count_b = np.zeros(n_words, dtype=float)
            odd_block_count = 0
            for block in feature_blocks:
                ratings = block['ratings']
                n_participants = ratings.shape[0]
                permutation = rng.permutation(n_participants)
                if n_participants % 2 == 1:
                    permutation = permutation[:-1]
                    odd_block_count += 1
                half_size = len(permutation) // 2
                half_a = ratings[permutation[:half_size]]
                half_b = ratings[permutation[half_size:]]
                word_indices = block['word_indices']
                sum_a[word_indices] += half_a.sum(axis=0)
                count_a[word_indices] += half_size
                sum_b[word_indices] += half_b.sum(axis=0)
                count_b[word_indices] += half_size
            if np.any(count_a == 0) or np.any(count_b == 0):
                raise ValueError(
                    f'The split-half vectors for {feature} do not cover all emotion words.'
                )
            vector_a = sum_a / count_a
            vector_b = sum_b / count_b
            r_half = float(np.corrcoef(vector_a, vector_b)[0, 1])
            r_sb = float(2 * r_half / (1 + r_half))
            r_half_values[iteration - 1] = r_half
            r_sb_values[iteration - 1] = r_sb
            iteration_rows.append(
                {
                    'Emotion Feature': feature,
                    'Iteration': iteration,
                    'r_half': r_half,
                    'r_SB': r_sb,
                    'Number of Rating Blocks': len(feature_blocks),
                    'Number of Odd-Sized Rating Blocks': odd_block_count
                }
            )
        mean_reliability = float(r_sb_values.mean())
        if mean_reliability < 0.7:
            reliability_band = '< 0.70'
        elif mean_reliability < 0.8:
            reliability_band = '0.70-0.79'
        elif mean_reliability < 0.9:
            reliability_band = '0.80-0.89'
        else:
            reliability_band = '>= 0.90'
        summary_rows.append(
            {
                'Emotion Feature': feature,
                'Mean Split-Half Reliability': mean_reliability,
                'Median Split-Half Reliability': float(np.median(r_sb_values)),
                'Split-Half Reliability SD': float(r_sb_values.std(ddof=1)),
                '2.5th Percentile': float(np.percentile(r_sb_values, 2.5)),
                '97.5th Percentile': float(np.percentile(r_sb_values, 97.5)),
                'Minimum': float(r_sb_values.min()),
                'Maximum': float(r_sb_values.max()),
                'Number of Rating Blocks': len(feature_blocks),
                'Repetitions': n_splits,
                'Reliability Band': reliability_band
            }
        )
    return (pd.DataFrame(iteration_rows), pd.DataFrame(summary_rows))


## Data preparation and combination-level statistics


In [ ]:
if not INPUT_FILE.exists():
    raise FileNotFoundError(f'Input file not found: {INPUT_FILE}')
raw_data = pd.read_excel(INPUT_FILE)
source_columns = ['sub_id', 'group', 'word_raw_text', 'dim_name', 'rating']
missing_columns = [column for column in source_columns if column not in raw_data.columns]
if missing_columns:
    raise ValueError(f'Missing required columns: {missing_columns}')
if raw_data[source_columns].isna().any().any():
    raise ValueError('The required columns contain missing values.')
data = raw_data[source_columns].rename(
    columns={
        'sub_id': 'participant_id',
        'group': 'rating_block',
        'word_raw_text': 'emotion_word',
        'dim_name': 'emotion_feature'
    }
).copy()
for column in ['participant_id', 'rating_block', 'emotion_word', 'emotion_feature']:
    data[column] = data[column].astype(str).str.strip()
data['rating'] = pd.to_numeric(data['rating'], errors='raise')
if not np.all(np.isclose(data['rating'], np.round(data['rating']))):
    raise ValueError('The rating column contains non-integer values.')
data['rating'] = data['rating'].astype(int)
WORDS = sorted(data['emotion_word'].unique())
FEATURES = sorted(data['emotion_feature'].unique())
duplicate_count = int(data.duplicated(['participant_id', 'emotion_word', 'emotion_feature']).sum())
invalid_rating_count = int((~data['rating'].between(RATING_MIN, RATING_MAX)).sum())
participant_block_counts = data.groupby('participant_id')['rating_block'].nunique()
combination_block_counts = data.groupby(['emotion_word', 'emotion_feature'])['rating_block'].nunique()
observed_combinations = data.groupby(['emotion_word', 'emotion_feature']).ngroups
assert len(WORDS) == EXPECTED_WORDS, f'Observed {len(WORDS)} emotion words; expected {EXPECTED_WORDS}.'
assert len(FEATURES) == EXPECTED_FEATURES, f'Observed {len(FEATURES)} emotion features; expected {EXPECTED_FEATURES}.'
assert observed_combinations == EXPECTED_WORDS * EXPECTED_FEATURES, 'The data do not cover all 70 x 43 emotion word-feature combinations.'
assert duplicate_count == 0, 'Duplicate participant-by-word-by-feature records were found.'
assert invalid_rating_count == 0, 'Ratings outside the 1-9 range were found.'
assert participant_block_counts.max() == 1, 'At least one participant appears in more than one rating block.'
assert combination_block_counts.max() == 1, 'At least one word-feature combination appears in more than one rating block.'
rating_block_design = data.groupby('rating_block').agg(
    **{
        'Number of Participants': ('participant_id', 'nunique'),
        'Number of Emotion Words': ('emotion_word', 'nunique'),
        'Number of Emotion Features': ('emotion_feature', 'nunique'),
        'Number of Rating Records': ('rating', 'size')
    }
).reset_index().rename(columns={'rating_block': 'Rating Block'})
observed_block_combinations = data[
    ['rating_block', 'emotion_word', 'emotion_feature']
].drop_duplicates().groupby('rating_block').size()
participant_rows = data.groupby(['rating_block', 'participant_id']).size()
rating_block_design['Expected Combinations'] = rating_block_design['Number of Emotion Words'] * rating_block_design['Number of Emotion Features']
rating_block_design['Observed Combinations'] = rating_block_design['Rating Block'].map(observed_block_combinations)
rating_block_design['Minimum Ratings per Participant'] = rating_block_design['Rating Block'].map(participant_rows.groupby(level=0).min())
rating_block_design['Maximum Ratings per Participant'] = rating_block_design['Rating Block'].map(participant_rows.groupby(level=0).max())
rating_block_design['Design Complete'] = (rating_block_design['Expected Combinations'] == rating_block_design['Observed Combinations']) & (
    rating_block_design['Expected Combinations'] == rating_block_design['Minimum Ratings per Participant']
) & (rating_block_design['Expected Combinations'] == rating_block_design['Maximum Ratings per Participant'])
assert rating_block_design['Design Complete'].all(), 'At least one rating block is incomplete.'
combination_statistics = data.groupby(['emotion_word', 'emotion_feature']).agg(
    **{
        'Rating Block': ('rating_block', 'first'),
        'Number of Raters': ('participant_id', 'nunique'),
        'Mean Rating': ('rating', 'mean'),
        'Rating SD': ('rating', 'std'),
        'Minimum Rating': ('rating', 'min'),
        'Maximum Rating': ('rating', 'max')
    }
).reset_index().rename(columns={'emotion_word': 'Emotion Word', 'emotion_feature': 'Emotion Feature'})
data_quality = pd.DataFrame(
    [
        ['Number of Rating Records', len(data)],
        ['Number of Participants', data['participant_id'].nunique()],
        ['Number of Rating Blocks', data['rating_block'].nunique()],
        ['Number of Emotion Words', len(WORDS)],
        ['Number of Emotion Features', len(FEATURES)],
        ['Observed Word-Feature Combinations', observed_combinations],
        ['Expected Word-Feature Combinations', EXPECTED_WORDS * EXPECTED_FEATURES],
        [
            'Missing Word-Feature Combinations',
            EXPECTED_WORDS * EXPECTED_FEATURES - observed_combinations
        ],
        [
            'Records Missing Required Values',
            int(
                data[
                    ['participant_id', 'rating_block', 'emotion_word', 'emotion_feature', 'rating']
                ].isna().any(axis=1).sum()
            )
        ],
        ['Duplicate Participant-Word-Feature Records', duplicate_count],
        ['Ratings Outside the Valid Range', invalid_rating_count],
        ['Minimum Rating', data['rating'].min()],
        ['Maximum Rating', data['rating'].max()],
        ['Minimum Raters per Combination', combination_statistics['Number of Raters'].min()],
        ['Maximum Raters per Combination', combination_statistics['Number of Raters'].max()],
        ['All Rating Blocks Complete', bool(rating_block_design['Design Complete'].all())]
    ],
    columns=['Check', 'Result']
)
save_csv(data_quality, PROCESS_DIR, '02_data_quality_checks.csv')
save_csv(rating_block_design, PROCESS_DIR, '03_rating_block_design_summary.csv')
save_csv(combination_statistics, PROCESS_DIR, '04_word_feature_combination_statistics.csv')


## Split-half reliability of the 43 emotion features


In [ ]:
reliability_blocks = build_reliability_blocks(data, WORDS)
split_half_iterations, reliability_summary = repeated_split_half(
    blocks=reliability_blocks,
    words=WORDS,
    features=FEATURES,
    seed=SEED,
    n_splits=N_SPLITS
)
overall_reliability = pd.DataFrame(
    [
        {
            'Sample': 'Main Sample',
            'Number of Emotion Features': len(FEATURES),
            'Mean Feature Reliability': reliability_summary['Mean Split-Half Reliability'].mean(),
            'Median Feature Reliability': reliability_summary['Mean Split-Half Reliability'].median(),
            'Minimum Feature Reliability': reliability_summary['Mean Split-Half Reliability'].min(),
            'Maximum Feature Reliability': reliability_summary['Mean Split-Half Reliability'].max(),
            'Features Below 0.70': int(
                (reliability_summary['Mean Split-Half Reliability'] < 0.7).sum()
            ),
            'Features at or Above 0.70': int(
                (reliability_summary['Mean Split-Half Reliability'] >= 0.7).sum()
            ),
            'Features at or Above 0.80': int(
                (reliability_summary['Mean Split-Half Reliability'] >= 0.8).sum()
            ),
            'Features at or Above 0.90': int(
                (reliability_summary['Mean Split-Half Reliability'] >= 0.9).sum()
            ),
            'Random Seed': SEED,
            'Repetitions': N_SPLITS
        }
    ]
)
save_csv(split_half_iterations, PROCESS_DIR, '05_split_half_iteration_results.csv')
save_csv(reliability_summary, RESULTS_DIR, '01_split_half_reliability_summary.csv')
save_csv(overall_reliability, RESULTS_DIR, '02_overall_reliability_summary.csv')


## Emotion-feature rating distributions


In [ ]:
word_feature_means = data.groupby(['emotion_word', 'emotion_feature']).agg(
    **{
        'Mean Rating': ('rating', 'mean'),
        'Rating SD': ('rating', 'std'),
        'Number of Raters': ('participant_id', 'nunique'),
        'Rating Block': ('rating_block', 'first')
    }
).reset_index().rename(columns={'emotion_word': 'Emotion Word', 'emotion_feature': 'Emotion Feature'})
mean_matrix = word_feature_means.pivot(
    index='Emotion Word',
    columns='Emotion Feature',
    values='Mean Rating'
).reindex(index=WORDS, columns=FEATURES)
if mean_matrix.isna().any().any():
    raise ValueError('The 70 x 43 mean-rating matrix contains missing values.')
description_rows = []
top_bottom_rows = []
for feature in FEATURES:
    scores = mean_matrix[feature]
    lowest = scores.nsmallest(TOP_N)
    highest = scores.nlargest(TOP_N)
    description_rows.append(
        {
            'Emotion Feature': feature,
            'Number of Emotion Words': int(scores.notna().sum()),
            'Mean Across Word-Level Means': float(scores.mean()),
            'SD Across Word-Level Means': float(scores.std(ddof=1)),
            'Minimum': float(scores.min()),
            'First Quartile': float(scores.quantile(0.25)),
            'Median': float(scores.median()),
            'Third Quartile': float(scores.quantile(0.75)),
            'Maximum': float(scores.max()),
            'Range': float(scores.max() - scores.min()),
            'Lowest-Rated Emotion Word': lowest.index[0],
            'Lowest Mean Rating': float(lowest.iloc[0]),
            'Highest-Rated Emotion Word': highest.index[0],
            'Highest Mean Rating': float(highest.iloc[0]),
            'Three Lowest-Rated Emotion Words': '; '.join(lowest.index.tolist()),
            'Three Highest-Rated Emotion Words': '; '.join(highest.index.tolist())
        }
    )
    for rank, (word, score) in enumerate(lowest.items(), start=1):
        n_raters = int(
            word_feature_means.loc[
                (word_feature_means['Emotion Word'] == word) & (
                    word_feature_means['Emotion Feature'] == feature
                ),
                'Number of Raters'
            ].iloc[0]
        )
        top_bottom_rows.append(
            {
                'Emotion Feature': feature,
                'Direction': 'Lowest',
                'Rank Within Direction': rank,
                'Emotion Word': word,
                'Mean Rating': float(score),
                'Number of Raters': n_raters
            }
        )
    for rank, (word, score) in enumerate(highest.items(), start=1):
        n_raters = int(
            word_feature_means.loc[
                (word_feature_means['Emotion Word'] == word) & (
                    word_feature_means['Emotion Feature'] == feature
                ),
                'Number of Raters'
            ].iloc[0]
        )
        top_bottom_rows.append(
            {
                'Emotion Feature': feature,
                'Direction': 'Highest',
                'Rank Within Direction': rank,
                'Emotion Word': word,
                'Mean Rating': float(score),
                'Number of Raters': n_raters
            }
        )
feature_description = pd.DataFrame(description_rows)
top_bottom_words = pd.DataFrame(top_bottom_rows)
save_csv(word_feature_means, RESULTS_DIR, '03_word_feature_means_long.csv')
save_csv(mean_matrix.reset_index(), RESULTS_DIR, '04_mean_rating_matrix.csv')
save_csv(feature_description, RESULTS_DIR, '05_feature_distribution_summary.csv')
save_csv(top_bottom_words, RESULTS_DIR, '06_feature_top_bottom_words.csv')


## Summary outputs


In [ ]:
summary_workbook = RESULTS_DIR / 'reliability_description_summary.xlsx'
with pd.ExcelWriter(summary_workbook, engine='openpyxl') as writer:
    data_quality.to_excel(writer, sheet_name='Data_QC', index=False)
    rating_block_design.to_excel(writer, sheet_name='Rating_Block_Design', index=False)
    overall_reliability.to_excel(writer, sheet_name='Overall_Reliability', index=False)
    reliability_summary.to_excel(writer, sheet_name='Split_Half_Reliability', index=False)
    word_feature_means.to_excel(writer, sheet_name='Word_Feature_Means', index=False)
    mean_matrix.reset_index().to_excel(writer, sheet_name='Mean_Rating_Matrix', index=False)
    feature_description.to_excel(writer, sheet_name='Feature_Distributions', index=False)
    top_bottom_words.to_excel(writer, sheet_name='Top_Bottom_Words', index=False)
